In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/iamaaron07/ddinters/DDInterTable (1) (1).csv
/kaggle/input/datasets/iamaaron07/patients/patient.csv
/kaggle/input/datasets/iamaaron07/admissions/admission.csv
/kaggle/input/datasets/iamaaron07/noteeventss/noteevents.csv


In [2]:
# ============================================================
# Import Libraries
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Load Datasets
# ============================================================

patient = pd.read_csv(
    "/kaggle/input/datasets/iamaaron07/patients/patient.csv"
)

admission = pd.read_csv(
    "/kaggle/input/datasets/iamaaron07/admissions/admission.csv"
)

noteevents = pd.read_csv(
    "/kaggle/input/datasets/iamaaron07/noteeventss/noteevents.csv"
)

ddinter = pd.read_csv(
    "/kaggle/input/datasets/iamaaron07/ddinters/DDInterTable (1) (1).csv"
)

# ============================================================
# Display Dataset Shapes
# ============================================================

print("Patient Table    :", patient.shape)
print("Admission Table  :", admission.shape)
print("Note Events Table:", noteevents.shape)



Patient Table    : (1196, 5)
Admission Table  : (1196, 7)
Note Events Table: (15197, 6)


In [3]:
# ==========================================================
# Dataset Summary (based on subject_id)
# ==========================================================

def dataset_summary(df, name):
    print("\n" + "="*80)
    print(f"DATASET : {name}")
    print("="*80)

    # Shape
    print(f"Shape                    : {df.shape}")

    # Unique subject IDs
    unique_subjects = df["subject_id"].nunique()
    print(f"Unique subject_ids       : {unique_subjects}")

    # Duplicate subject IDs
    duplicate_subjects = df["subject_id"].duplicated().sum()
    print(f"Duplicate subject_ids    : {duplicate_subjects}")

    # Total null values
    total_nulls = df.isnull().sum().sum()
    print(f"Total Null Values        : {total_nulls}")

    # Top 10 repeated subject IDs
    print("\nTop 10 Repeated subject_ids:")

    top_duplicates = (
        df["subject_id"]
        .value_counts()
        .reset_index()
    )

    top_duplicates.columns = ["subject_id", "Count"]

    display(top_duplicates.head(10))


# ==========================================================
# Run for datasets
# ==========================================================

dataset_summary(patient, "PATIENT")
dataset_summary(admission, "ADMISSION")
dataset_summary(noteevents, "NOTEEVENTS")


DATASET : PATIENT
Shape                    : (1196, 5)
Unique subject_ids       : 1099
Duplicate subject_ids    : 97
Total Null Values        : 0

Top 10 Repeated subject_ids:


,subject_id,Count
0,9911d9251,4
1,8632aa8bd,4
2,402219128,3
3,c1dffee5d,2
4,265641410,2
5,8c3783352,2
6,87fa853ac,2
7,46a555916,2
8,2b746242f,2
9,3e377ab72,2



DATASET : ADMISSION
Shape                    : (1196, 7)
Unique subject_ids       : 1099
Duplicate subject_ids    : 97
Total Null Values        : 0

Top 10 Repeated subject_ids:


,subject_id,Count
0,9911d9251,4
1,8632aa8bd,4
2,402219128,3
3,c1dffee5d,2
4,265641410,2
5,8c3783352,2
6,87fa853ac,2
7,46a555916,2
8,2b746242f,2
9,3e377ab72,2



DATASET : NOTEEVENTS
Shape                    : (15197, 6)
Unique subject_ids       : 1687
Duplicate subject_ids    : 13510
Total Null Values        : 9566

Top 10 Repeated subject_ids:


,subject_id,Count
0,f8095f49f,286
1,77f6e5997,227
2,ff9c68c65,139
3,1d78013d7,110
4,dbb1c817a,101
5,a7c8bffbc,97
6,e2ead134d,86
7,d6d85a4b0,84
8,42ddfadfe,84
9,69f30d37e,83


In [4]:
noteevents.isnull().sum()

row_id                    0
subject_id                0
chart_date              155
diagnosis                 0
respiratory_support    9411
description               0
dtype: int64

In [5]:
# ==========================================
# Remove rows with missing chart_date
# ==========================================

print("Shape before:", noteevents.shape)

# Remove rows where chart_date is null
noteevents = noteevents.dropna(subset=["chart_date"]).reset_index(drop=True)

print("Shape after :", noteevents.shape)
print("Rows removed:", 15197 - len(noteevents))

# Verify
print("\nRemaining null values:")
print(noteevents.isnull().sum())

Shape before: (15197, 6)
Shape after : (15042, 6)
Rows removed: 155

Remaining null values:
row_id                    0
subject_id                0
chart_date                0
diagnosis                 0
respiratory_support    9265
description               0
dtype: int64


In [6]:
print("DDInter Shape:", ddinter.shape)

DDInter Shape: (444766, 5)


In [7]:
# ============================================================
# Prepare DDInter Table
# ============================================================

ddinter["Drug_A"] = (
    ddinter["Drug_A"]
    .astype(str)
    .str.lower()
    .str.strip()
)

ddinter["Drug_B"] = (
    ddinter["Drug_B"]
    .astype(str)
    .str.lower()
    .str.strip()
)

ddinter["Level"] = (
    ddinter["Level"]
    .astype(str)
    .str.title()
)

print("DDInter Prepared")
display(ddinter.head())

DDInter Prepared


,DDInterID_A,Drug_A,DDInterID_B,Drug_B,Level
0,DDInter1263,naltrexone,DDInter1,abacavir,Moderate
1,DDInter1,abacavir,DDInter1348,orlistat,Moderate
2,DDInter58,aluminum hydroxide,DDInter582,dolutegravir,Major
3,DDInter112,aprepitant,DDInter582,dolutegravir,Minor
4,DDInter138,attapulgite,DDInter582,dolutegravir,Major


In [8]:
# ============================================================
# Build DDI Lookup Dictionary
# ============================================================

ddi_lookup = {}

for _, row in ddinter.iterrows():

    key = tuple(sorted([row["Drug_A"], row["Drug_B"]]))

    ddi_lookup[key] = row["Level"]

print("Total Known DDIs :", len(ddi_lookup))

Total Known DDIs : 160235


In [9]:
duplicates = (
    ddinter
    .groupby(["Drug_A", "Drug_B"])["Level"]
    .nunique()
)

print("Drug pairs with multiple severity levels:",
      (duplicates > 1).sum())

Drug pairs with multiple severity levels: 0


In [10]:
# ============================================================
# Create Drug List
# ============================================================

drug_list = sorted(
    list(
        set(ddinter["Drug_A"]) |
        set(ddinter["Drug_B"])
    )
)

print("Total drugs:", len(drug_list))

Total drugs: 1939


In [11]:
# ============================================================
# STEP 4 - Extract Drugs Mentioned in Clinical Notes
# ============================================================

records = []

for drug in drug_list:

    mask = noteevents["description"].str.contains(
        drug,
        case=False,
        na=False,
        regex=False
    )

    if mask.any():

        temp = noteevents.loc[
            mask,
            ["subject_id", "chart_date"]
        ].copy()

        temp["drug_name"] = drug

        records.append(temp)

drug_mentions_df = pd.concat(
    records,
    ignore_index=True
)

drug_mentions_df = (
    drug_mentions_df
    .drop_duplicates()
    .sort_values(
        ["subject_id", "chart_date", "drug_name"]
    )
    .reset_index(drop=True)
)

print("Total drug mentions:", len(drug_mentions_df))
display(drug_mentions_df.head())

Total drug mentions: 82259


,subject_id,chart_date,drug_name
0,00355bdbb,2134-12-09,fentanyl
1,00355bdbb,2134-12-09,lactulose
2,00355bdbb,2134-12-09,lorazepam
3,00355bdbb,2134-12-09,methylprednisolone
4,00355bdbb,2134-12-09,piperacillin


In [12]:
# ============================================================
# STEP 5A - Create Drug List for Each Patient-Day
# ============================================================

drug_used_df = (
    drug_mentions_df
    .groupby(["subject_id", "chart_date"])["drug_name"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)

drug_used_df.rename(columns={"drug_name": "drug_used"}, inplace=True)

print("Total Patient-Day Records:", len(drug_used_df))
display(drug_used_df.head())

Total Patient-Day Records: 14546


,subject_id,chart_date,drug_used
0,00355bdbb,2134-12-09,"[fentanyl, lactulose, lorazepam, methylprednis..."
1,00355bdbb,2134-12-10,"[fentanyl, lactulose, lorazepam, methylprednis..."
2,00355bdbb,2134-12-11,"[fentanyl, lactulose, lorazepam, methylprednis..."
3,00355bdbb,2134-12-12,"[calcium gluconate, fentanyl, lactulose, loraz..."
4,00355bdbb,2134-12-13,"[fentanyl, lactulose, lorazepam, meropenem, os..."


In [13]:
# ============================================================
# STEP 5B - Generate All Drug Combinations
# ============================================================

from itertools import combinations

drug_used_df["drug_combinations"] = drug_used_df["drug_used"].apply(
    lambda drugs: [tuple(sorted(pair)) for pair in combinations(drugs, 2)]
)

display(drug_used_df.head())

,subject_id,chart_date,drug_used,drug_combinations
0,00355bdbb,2134-12-09,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, lorazepam),..."
1,00355bdbb,2134-12-10,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, lorazepam),..."
2,00355bdbb,2134-12-11,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, lorazepam),..."
3,00355bdbb,2134-12-12,"[calcium gluconate, fentanyl, lactulose, loraz...","[(calcium gluconate, fentanyl), (calcium gluco..."
4,00355bdbb,2134-12-13,"[fentanyl, lactulose, lorazepam, meropenem, os...","[(fentanyl, lactulose), (fentanyl, lorazepam),..."


In [14]:
# ============================================================
# STEP 5C - Keep Only Matched Drug Combinations
# ============================================================

def get_matched_pairs(pair_list):
    
    matched = []

    for pair in pair_list:

        key = tuple(sorted(pair))

        if key in ddi_lookup:
            matched.append(pair)

    return matched


drug_used_df["matched_ddi_pairs"] = (
    drug_used_df["drug_combinations"]
    .apply(get_matched_pairs)
)

display(drug_used_df.head())

,subject_id,chart_date,drug_used,drug_combinations,matched_ddi_pairs
0,00355bdbb,2134-12-09,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, lorazepam),...","[(fentanyl, lactulose), (fentanyl, methylpredn..."
1,00355bdbb,2134-12-10,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, lorazepam),...","[(fentanyl, lactulose), (fentanyl, methylpredn..."
2,00355bdbb,2134-12-11,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, lorazepam),...","[(fentanyl, lactulose), (fentanyl, methylpredn..."
3,00355bdbb,2134-12-12,"[calcium gluconate, fentanyl, lactulose, loraz...","[(calcium gluconate, fentanyl), (calcium gluco...","[(fentanyl, lactulose), (fentanyl, methylpredn..."
4,00355bdbb,2134-12-13,"[fentanyl, lactulose, lorazepam, meropenem, os...","[(fentanyl, lactulose), (fentanyl, lorazepam),...","[(fentanyl, lactulose), (fentanyl, prednisolon..."


In [15]:
# ============================================================
# STEP 5D - Count Total DDIs
# ============================================================

drug_used_df["total_ddi"] = (
    drug_used_df["matched_ddi_pairs"]
    .apply(len)
)

display(drug_used_df.head())

,subject_id,chart_date,drug_used,drug_combinations,matched_ddi_pairs,total_ddi
0,00355bdbb,2134-12-09,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, lorazepam),...","[(fentanyl, lactulose), (fentanyl, methylpredn...",12
1,00355bdbb,2134-12-10,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, lorazepam),...","[(fentanyl, lactulose), (fentanyl, methylpredn...",12
2,00355bdbb,2134-12-11,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, lorazepam),...","[(fentanyl, lactulose), (fentanyl, methylpredn...",15
3,00355bdbb,2134-12-12,"[calcium gluconate, fentanyl, lactulose, loraz...","[(calcium gluconate, fentanyl), (calcium gluco...","[(fentanyl, lactulose), (fentanyl, methylpredn...",15
4,00355bdbb,2134-12-13,"[fentanyl, lactulose, lorazepam, meropenem, os...","[(fentanyl, lactulose), (fentanyl, lorazepam),...","[(fentanyl, lactulose), (fentanyl, prednisolon...",9


In [16]:
# ============================================================
# Remove unwanted columns
# ============================================================

drug_used_df = drug_used_df.drop(
    columns=["drug_combinations", "total_ddi"]
)

display(drug_used_df.head())

,subject_id,chart_date,drug_used,matched_ddi_pairs
0,00355bdbb,2134-12-09,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, methylpredn..."
1,00355bdbb,2134-12-10,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, methylpredn..."
2,00355bdbb,2134-12-11,"[fentanyl, lactulose, lorazepam, methylprednis...","[(fentanyl, lactulose), (fentanyl, methylpredn..."
3,00355bdbb,2134-12-12,"[calcium gluconate, fentanyl, lactulose, loraz...","[(fentanyl, lactulose), (fentanyl, methylpredn..."
4,00355bdbb,2134-12-13,"[fentanyl, lactulose, lorazepam, meropenem, os...","[(fentanyl, lactulose), (fentanyl, prednisolon..."


In [17]:
# ============================================================
# Create Patient-Day DDI Summary
# ============================================================

from collections import Counter

def summarize_ddis(pairs):

    major = []
    moderate = []
    minor = []

    for pair in pairs:

        severity = ddi_lookup.get(tuple(sorted(pair)))

        if severity == "Major":
            major.append(pair)

        elif severity == "Moderate":
            moderate.append(pair)

        elif severity == "Minor":
            minor.append(pair)

    return pd.Series({
        "ddi_major": major,
        "num_major": len(major),

        "ddi_moderate": moderate,
        "num_moderate": len(moderate),

        "ddi_minor": minor,
        "num_minor": len(minor)
    })


# Apply to every patient-day
severity_df = drug_used_df["matched_ddi_pairs"].apply(summarize_ddis)

# Merge with existing dataframe
patient_day_ddi = pd.concat(
    [
        drug_used_df.drop(columns=["matched_ddi_pairs"]),
        severity_df
    ],
    axis=1
)

print("Patient-Day DDI Summary Shape:", patient_day_ddi.shape)

display(patient_day_ddi.head())

Patient-Day DDI Summary Shape: (14546, 9)


,subject_id,chart_date,drug_used,ddi_major,num_major,ddi_moderate,num_moderate,ddi_minor,num_minor
0,00355bdbb,2134-12-09,"[fentanyl, lactulose, lorazepam, methylprednis...",[],0,"[(lactulose, methylprednisolone), (lactulose, ...",2,[],0
1,00355bdbb,2134-12-10,"[fentanyl, lactulose, lorazepam, methylprednis...",[],0,"[(lactulose, methylprednisolone), (lactulose, ...",2,[],0
2,00355bdbb,2134-12-11,"[fentanyl, lactulose, lorazepam, methylprednis...",[],0,"[(lactulose, methylprednisolone), (lactulose, ...",2,[],0
3,00355bdbb,2134-12-12,"[calcium gluconate, fentanyl, lactulose, loraz...",[],0,"[(lactulose, methylprednisolone), (lactulose, ...",2,[],0
4,00355bdbb,2134-12-13,"[fentanyl, lactulose, lorazepam, meropenem, os...",[],0,"[(lactulose, prednisolone)]",1,[],0


In [18]:
# ============================================================
# Remove Duplicate Admissions
# ============================================================

before = len(admission)

admission = admission.drop_duplicates(
    subset=["subject_id", "admitdate", "dischdate"],
    keep="first"
).reset_index(drop=True)

after = len(admission)

print("Rows before :", before)
print("Rows after  :", after)
print("Duplicates removed :", before - after)

Rows before : 1196
Rows after  : 1111
Duplicates removed : 85


In [19]:
# ============================================================
# STEP 9 - Match Patient-Day DDI with Correct Admission
# ============================================================

# Make sure dates are datetime
patient_day_ddi["chart_date"] = pd.to_datetime(patient_day_ddi["chart_date"])

admission["admitdate"] = pd.to_datetime(admission["admitdate"])
admission["dischdate"] = pd.to_datetime(admission["dischdate"])

# ------------------------------------------------------------
# Merge on subject_id
# ------------------------------------------------------------

patient_admission = patient_day_ddi.merge(
    admission,
    on="subject_id",
    how="left"
)

# ------------------------------------------------------------
# Keep only admissions where
# admitdate <= chart_date <= dischdate
# ------------------------------------------------------------

patient_admission = patient_admission[
    (patient_admission["chart_date"] >= patient_admission["admitdate"]) &
    (patient_admission["chart_date"] < patient_admission["dischdate"])
].copy()
patient_admission.reset_index(drop=True, inplace=True)

print("Matched Patient-Day Records:", patient_admission.shape)

display(patient_admission.head())

Matched Patient-Day Records: (6941, 15)


,subject_id,chart_date,drug_used,ddi_major,num_major,ddi_moderate,num_moderate,ddi_minor,num_minor,row_id,admitdate,admittime,dischdate,dischtime,release_description
0,0124e982f,2200-12-28,"[acyclovir, ceftriaxone, fentanyl, levetiracet...",[],0,[],0,[],0,1168.0,2200-12-28,04:18:45,2201-01-14,21:33:00,discharged
1,0124e982f,2200-12-29,"[acyclovir, ceftriaxone, fentanyl, levetiracet...",[],0,[],0,[],0,1168.0,2200-12-28,04:18:45,2201-01-14,21:33:00,discharged
2,0124e982f,2200-12-30,"[acyclovir, ceftriaxone, fentanyl, levetiracet...",[],0,[],0,[],0,1168.0,2200-12-28,04:18:45,2201-01-14,21:33:00,discharged
3,0124e982f,2200-12-31,"[acyclovir, ceftriaxone, fentanyl, levetiracet...",[],0,[],0,[],0,1168.0,2200-12-28,04:18:45,2201-01-14,21:33:00,discharged
4,0124e982f,2201-01-01,"[acyclovir, ceftriaxone, fentanyl, levetiracet...",[],0,[],0,[],0,1168.0,2200-12-28,04:18:45,2201-01-14,21:33:00,discharged


In [20]:
# Check if Subject ID + Chart Date is unique

duplicates = patient_admission.duplicated(
    subset=["subject_id", "chart_date"],
    keep=False
)

if duplicates.any():
    print("❌ Duplicate Subject ID + Chart Date found.")
    print("Total duplicate rows:", duplicates.sum())

    display(
        patient_admission[duplicates]
        .sort_values(["subject_id", "chart_date"])
    )
else:
    print("✅ No duplicates found.")

✅ No duplicates found.


In [21]:
wrong = patient_admission[
    (patient_admission["chart_date"] < patient_admission["admitdate"]) |
    (patient_admission["chart_date"] > patient_admission["dischdate"])
]

print("Wrong rows:", len(wrong))
display(wrong.head())

Wrong rows: 0


,subject_id,chart_date,drug_used,ddi_major,num_major,ddi_moderate,num_moderate,ddi_minor,num_minor,row_id,admitdate,admittime,dischdate,dischtime,release_description


In [22]:
# Add number of drugs
patient_admission["num_drugs"] = patient_admission["drug_used"].apply(len)

print(patient_admission[["drug_used", "num_drugs"]].head())

                                           drug_used  num_drugs
0  [acyclovir, ceftriaxone, fentanyl, levetiracet...          6
1  [acyclovir, ceftriaxone, fentanyl, levetiracet...          7
2  [acyclovir, ceftriaxone, fentanyl, levetiracet...          6
3  [acyclovir, ceftriaxone, fentanyl, levetiracet...          6
4  [acyclovir, ceftriaxone, fentanyl, levetiracet...          6


In [23]:
# ============================================================
# STEP 10 - Calculate Length of Stay (LOS)
# ============================================================

# Ensure dates are datetime
patient_admission["admitdate"] = pd.to_datetime(patient_admission["admitdate"])
patient_admission["dischdate"] = pd.to_datetime(patient_admission["dischdate"])

# Calculate LOS in days
patient_admission["LOS_days"] = (
    patient_admission["dischdate"] - patient_admission["admitdate"]
).dt.days

print(patient_admission.head())

  subject_id chart_date                                          drug_used  \
0  0124e982f 2200-12-28  [acyclovir, ceftriaxone, fentanyl, levetiracet...   
1  0124e982f 2200-12-29  [acyclovir, ceftriaxone, fentanyl, levetiracet...   
2  0124e982f 2200-12-30  [acyclovir, ceftriaxone, fentanyl, levetiracet...   
3  0124e982f 2200-12-31  [acyclovir, ceftriaxone, fentanyl, levetiracet...   
4  0124e982f 2201-01-01  [acyclovir, ceftriaxone, fentanyl, levetiracet...   

  ddi_major  num_major ddi_moderate  num_moderate ddi_minor  num_minor  \
0        []          0           []             0        []          0   
1        []          0           []             0        []          0   
2        []          0           []             0        []          0   
3        []          0           []             0        []          0   
4        []          0           []             0        []          0   

   row_id  admitdate admittime  dischdate dischtime release_description  \
0  1168.0 2

In [24]:
# Remove admission and discharge columns

patient_admission = patient_admission.drop(
    columns=[
        "admitdate",
        "admittime",
        "dischdate",
        "dischtime"
    ]
)

print("Shape:", patient_admission.shape)
display(patient_admission.head())

Shape: (6941, 13)


,subject_id,chart_date,drug_used,ddi_major,num_major,ddi_moderate,num_moderate,ddi_minor,num_minor,row_id,release_description,num_drugs,LOS_days
0,0124e982f,2200-12-28,"[acyclovir, ceftriaxone, fentanyl, levetiracet...",[],0,[],0,[],0,1168.0,discharged,6,17
1,0124e982f,2200-12-29,"[acyclovir, ceftriaxone, fentanyl, levetiracet...",[],0,[],0,[],0,1168.0,discharged,7,17
2,0124e982f,2200-12-30,"[acyclovir, ceftriaxone, fentanyl, levetiracet...",[],0,[],0,[],0,1168.0,discharged,6,17
3,0124e982f,2200-12-31,"[acyclovir, ceftriaxone, fentanyl, levetiracet...",[],0,[],0,[],0,1168.0,discharged,6,17
4,0124e982f,2201-01-01,"[acyclovir, ceftriaxone, fentanyl, levetiracet...",[],0,[],0,[],0,1168.0,discharged,6,17


In [25]:
# ============================================
# Rename release_description -> mortality
# Convert discharged = 0, expired = 1
# ============================================

patient_admission = patient_admission.rename(
    columns={"release_description": "mortality"}
)

patient_admission["mortality"] = (
    patient_admission["mortality"]
    .str.lower()
    .map({
        "discharged": 0,
        "expired": 1
    })
)

print("Shape:", patient_admission.shape)

# Show all columns
print("\nColumns:")
print(patient_admission.columns.tolist())

# Display first 5 rows with all columns
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None
):
    display(patient_admission.head())

Shape: (6941, 13)

Columns:
['subject_id', 'chart_date', 'drug_used', 'ddi_major', 'num_major', 'ddi_moderate', 'num_moderate', 'ddi_minor', 'num_minor', 'row_id', 'mortality', 'num_drugs', 'LOS_days']


,subject_id,chart_date,drug_used,ddi_major,num_major,ddi_moderate,num_moderate,ddi_minor,num_minor,row_id,mortality,num_drugs,LOS_days
0,0124e982f,2200-12-28,"[acyclovir, ceftriaxone, fentanyl, levetiracetam, mannitol, midazolam]",[],0,[],0,[],0,1168.0,0.0,6,17
1,0124e982f,2200-12-29,"[acyclovir, ceftriaxone, fentanyl, levetiracetam, mannitol, midazolam, vitamin d]",[],0,[],0,[],0,1168.0,0.0,7,17
2,0124e982f,2200-12-30,"[acyclovir, ceftriaxone, fentanyl, levetiracetam, midazolam, vitamin d]",[],0,[],0,[],0,1168.0,0.0,6,17
3,0124e982f,2200-12-31,"[acyclovir, ceftriaxone, fentanyl, levetiracetam, midazolam, vitamin d]",[],0,[],0,[],0,1168.0,0.0,6,17
4,0124e982f,2201-01-01,"[acyclovir, ceftriaxone, fentanyl, levetiracetam, midazolam, vitamin d]",[],0,[],0,[],0,1168.0,0.0,6,17


In [26]:
# ============================================
# Add Age and Gender using subject_id + row_id
# ============================================

# Keep only required columns
patient_info = patient[
    ["row_id", "subject_id", "age_years", "gender"]
].copy()

# Merge on both keys
patient_admission = patient_admission.merge(
    patient_info,
    on=["row_id", "subject_id"],
    how="left",
    validate="many_to_one"   # many patient_admission rows -> one patient row
)

# Rename age column
patient_admission.rename(
    columns={"age_years": "age"},
    inplace=True
)

print("Shape:", patient_admission.shape)

print("\nMissing Age:", patient_admission["age"].isna().sum())
print("Missing Gender:", patient_admission["gender"].isna().sum())

print("\nColumns:")
print(patient_admission.columns.tolist())

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None
):
    display(patient_admission.head())

Shape: (6941, 15)

Missing Age: 0
Missing Gender: 0

Columns:
['subject_id', 'chart_date', 'drug_used', 'ddi_major', 'num_major', 'ddi_moderate', 'num_moderate', 'ddi_minor', 'num_minor', 'row_id', 'mortality', 'num_drugs', 'LOS_days', 'age', 'gender']


,subject_id,chart_date,drug_used,ddi_major,num_major,ddi_moderate,num_moderate,ddi_minor,num_minor,row_id,mortality,num_drugs,LOS_days,age,gender
0,0124e982f,2200-12-28,"[acyclovir, ceftriaxone, fentanyl, levetiracetam, mannitol, midazolam]",[],0,[],0,[],0,1168.0,0.0,6,17,5.19,Female
1,0124e982f,2200-12-29,"[acyclovir, ceftriaxone, fentanyl, levetiracetam, mannitol, midazolam, vitamin d]",[],0,[],0,[],0,1168.0,0.0,7,17,5.19,Female
2,0124e982f,2200-12-30,"[acyclovir, ceftriaxone, fentanyl, levetiracetam, midazolam, vitamin d]",[],0,[],0,[],0,1168.0,0.0,6,17,5.19,Female
3,0124e982f,2200-12-31,"[acyclovir, ceftriaxone, fentanyl, levetiracetam, midazolam, vitamin d]",[],0,[],0,[],0,1168.0,0.0,6,17,5.19,Female
4,0124e982f,2201-01-01,"[acyclovir, ceftriaxone, fentanyl, levetiracetam, midazolam, vitamin d]",[],0,[],0,[],0,1168.0,0.0,6,17,5.19,Female


In [27]:
# ============================================================
# Patients With and Without At Least One DDI
# ============================================================

# Total DDIs for each patient-day
patient_admission["total_ddi"] = (
    patient_admission["num_major"] +
    patient_admission["num_moderate"] +
    patient_admission["num_minor"]
)

# Maximum DDI count observed for each patient
patient_summary = (
    patient_admission
    .groupby("subject_id")["total_ddi"]
    .max()
    .reset_index()
)

patients_with_ddi = (patient_summary["total_ddi"] > 0).sum()
patients_without_ddi = (patient_summary["total_ddi"] == 0).sum()

print("Total Unique Patients           :", len(patient_summary))
print("Patients With At Least One DDI  :", patients_with_ddi)
print("Patients With No DDI            :", patients_without_ddi)

Total Unique Patients           : 709
Patients With At Least One DDI  : 435
Patients With No DDI            : 274


In [28]:
# Save the final dataset
patient_admission.to_csv(
    "/kaggle/working/final_patient_admission_dataset.csv",
    index=False
)

print("Dataset saved successfully!")
print("Final Shape:", patient_admission.shape)

Dataset saved successfully!
Final Shape: (6941, 16)


In [29]:
# Count males and females
gender_counts = patient_admission["gender"].value_counts()

print(gender_counts)

gender
Male      4275
Female    2666
Name: count, dtype: int64


In [30]:
# ============================================================
# Number of Unique Patients
# ============================================================

unique_patients = patient_admission["subject_id"].drop_duplicates()

print("Total Unique Patients:", len(unique_patients))

Total Unique Patients: 709


In [31]:
# ============================================================
# Number of Prescriptions per Patient
# ============================================================

prescriptions_per_patient = (
    patient_admission
    .groupby("subject_id")["chart_date"]
    .nunique()
    .reset_index(name="num_prescriptions")
)

print("First 10 Patients:")
display(prescriptions_per_patient.head(10))

print("\nSummary Statistics")
print(prescriptions_per_patient["num_prescriptions"].describe())


First 10 Patients:


,subject_id,num_prescriptions
0,0124e982f,11
1,013599d68,5
2,01b918381,15
3,023a49496,5
4,028a2b3e8,6
5,028fe7e69,14
6,030b82fa4,1
7,035adc181,2
8,045018263,7
9,0453b0d28,6



Summary Statistics
count    709.000000
mean       9.789845
std       16.151284
min        1.000000
25%        2.000000
50%        5.000000
75%       12.000000
max      286.000000
Name: num_prescriptions, dtype: float64


In [32]:
# Total number of prescriptions across all patients
total_prescriptions = prescriptions_per_patient["num_prescriptions"].sum()

print("Total Prescriptions:", total_prescriptions)

Total Prescriptions: 6941


In [33]:
duplicate_prescriptions = patient_admission.duplicated(
    subset=["subject_id", "chart_date"]
).sum()

print("Duplicate subject_id + chart_date:", duplicate_prescriptions)

Duplicate subject_id + chart_date: 0


In [34]:
# ============================================================
# Number of Unique Potential DDIs
# ============================================================

unique_ddis = set()

for _, row in patient_admission.iterrows():

    # Major DDIs
    for pair in row["ddi_major"]:
        unique_ddis.add(tuple(sorted(pair)))

    # Moderate DDIs
    for pair in row["ddi_moderate"]:
        unique_ddis.add(tuple(sorted(pair)))

    # Minor DDIs
    for pair in row["ddi_minor"]:
        unique_ddis.add(tuple(sorted(pair)))

print("Total Unique pDDIs:", len(unique_ddis))

Total Unique pDDIs: 403


In [35]:
# ============================================================
# Number of Patients With At Least One DDI
# and Number of Patients With No DDI
# ============================================================

# Total DDIs on each prescription (patient-day)
patient_admission["total_ddi"] = (
    patient_admission["num_major"] +
    patient_admission["num_moderate"] +
    patient_admission["num_minor"]
)

# For each patient, find the maximum DDI count across all chart dates
patient_summary = (
    patient_admission
    .groupby("subject_id")["total_ddi"]
    .max()
    .reset_index()
)

# Patients with at least one DDI
patients_with_ddi = (patient_summary["total_ddi"] > 0).sum()

# Patients with no DDI
patients_without_ddi = (patient_summary["total_ddi"] == 0).sum()

# Total unique patients
total_patients = len(patient_summary)

print("Total Unique Patients           :", total_patients)
print("Patients With At Least One DDI  :", patients_with_ddi)
print("Patients With No DDI            :", patients_without_ddi)

Total Unique Patients           : 709
Patients With At Least One DDI  : 435
Patients With No DDI            : 274


In [36]:
# ============================================================
# Age Analysis (Unique Ages per Subject)
# ============================================================

# Maximum age
max_age = patient_admission["age"].max()

# Minimum age
min_age = patient_admission["age"].min()

# Keep only unique (subject_id, age) combinations
unique_patient_age = patient_admission[
    ["subject_id", "age"]
].drop_duplicates()

# Median age from the resulting sample
median_age = unique_patient_age["age"].median()

print(f"Minimum Age : {min_age:.2f} years")
print(f"Maximum Age : {max_age:.2f} years")
print(f"Median Age  : {median_age:.2f} years")

print("\nNumber of Unique (Subject ID, Age) Records:",
      len(unique_patient_age))

Minimum Age : 0.17 years
Maximum Age : 18.85 years
Median Age  : 5.36 years

Number of Unique (Subject ID, Age) Records: 710


In [37]:
# ============================================================
# Total DDI Occurrences (Chart-Date Wise)
# ============================================================

total_major = patient_admission["num_major"].sum()
total_moderate = patient_admission["num_moderate"].sum()
total_minor = patient_admission["num_minor"].sum()

total_ddi = total_major + total_moderate + total_minor

print("Major DDIs     :", total_major)
print("Moderate DDIs :", total_moderate)
print("Minor DDIs     :", total_minor)
print("-" * 35)
print("TOTAL DDI OCCURRENCES :", total_ddi)

Major DDIs     : 1679
Moderate DDIs : 6807
Minor DDIs     : 921
-----------------------------------
TOTAL DDI OCCURRENCES : 9407


In [38]:
# ============================================================
# Top 20 Most Frequent Drugs (Chart-Date Wise)
# ============================================================

from collections import Counter

drug_counter = Counter()

for drugs in patient_admission["drug_used"]:
    drug_counter.update(drugs)

top20_drugs = (
    pd.DataFrame(
        drug_counter.items(),
        columns=["Drug","Frequency"]
    )
    .sort_values("Frequency", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

display(top20_drugs)

,Drug,Frequency
0,fentanyl,4054
1,midazolam,3414
2,vancomycin,2022
3,chlorhexidine,1505
4,meropenem,1366
5,amikacin,1264
6,vitamin d,1204
7,pantoprazole,1079
8,ceftriaxone,988
9,piperacillin,855


In [39]:
# ============================================================
# Top 20 Most Frequent Drug Interactions
# ============================================================

ddi_counter = Counter()

for col in ["ddi_major","ddi_moderate","ddi_minor"]:
    for interactions in patient_admission[col]:
        ddi_counter.update(
            [tuple(sorted(pair)) for pair in interactions]
        )

top20_ddi = (
    pd.DataFrame(
        ddi_counter.items(),
        columns=["Drug Pair","Frequency"]
    )
    .sort_values("Frequency", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

display(top20_ddi)

,Drug Pair,Frequency
0,"(dobutamine, vancomycin)",315
1,"(piperacillin, vancomycin)",298
2,"(amikacin, pantoprazole)",238
3,"(amikacin, piperacillin)",233
4,"(dexamethasone, fentanyl)",221
5,"(dexamethasone, midazolam)",213
6,"(amikacin, cefotaxime)",200
7,"(furosemide, salbutamol)",180
8,"(vancomycin, vecuronium)",176
9,"(azithromycin, salbutamol)",167


In [40]:
# ============================================================
# Top 10 Major DDIs
# ============================================================

major_counter = Counter()

for interactions in patient_admission["ddi_major"]:
    major_counter.update(
        [tuple(sorted(pair)) for pair in interactions]
    )

top10_major = (
    pd.DataFrame(
        major_counter.items(),
        columns=["Drug Pair","Frequency"]
    )
    .sort_values("Frequency", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(top10_major)

,Drug Pair,Frequency
0,"(piperacillin, vancomycin)",298
1,"(dexamethasone, fentanyl)",221
2,"(fentanyl, fluconazole)",131
3,"(fluconazole, midazolam)",112
4,"(dexamethasone, ofloxacin)",86
5,"(amikacin, furosemide)",85
6,"(amikacin, vecuronium)",74
7,"(calcium gluconate, ceftriaxone)",73
8,"(dexamethasone, levofloxacin)",72
9,"(fentanyl, ondansetron)",48


In [41]:
# ============================================================
# Top 10 Moderate DDIs
# ============================================================

moderate_counter = Counter()

for interactions in patient_admission["ddi_moderate"]:
    moderate_counter.update(
        [tuple(sorted(pair)) for pair in interactions]
    )

top10_moderate = (
    pd.DataFrame(
        moderate_counter.items(),
        columns=["Drug Pair","Frequency"]
    )
    .sort_values("Frequency", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(top10_moderate)

,Drug Pair,Frequency
0,"(dobutamine, vancomycin)",315
1,"(amikacin, pantoprazole)",238
2,"(amikacin, piperacillin)",233
3,"(amikacin, cefotaxime)",200
4,"(furosemide, salbutamol)",180
5,"(vancomycin, vecuronium)",176
6,"(azithromycin, salbutamol)",167
7,"(budesonide, metoprolol)",159
8,"(dopamine, vancomycin)",144
9,"(amphotericin b, vancomycin)",134


In [42]:
# ============================================================
# Top 10 Minor DDIs
# ============================================================

minor_counter = Counter()

for interactions in patient_admission["ddi_minor"]:
    minor_counter.update(
        [tuple(sorted(pair)) for pair in interactions]
    )

top10_minor = (
    pd.DataFrame(
        minor_counter.items(),
        columns=["Drug Pair","Frequency"]
    )
    .sort_values("Frequency", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(top10_minor)

,Drug Pair,Frequency
0,"(dexamethasone, midazolam)",213
1,"(budesonide, salbutamol)",156
2,"(midazolam, prednisolone)",146
3,"(methylprednisolone, midazolam)",71
4,"(hydrocortisone, salbutamol)",51
5,"(prednisolone, salbutamol)",33
6,"(dexamethasone, isoniazid)",27
7,"(methylprednisolone, salbutamol)",21
8,"(ceftriaxone, heparin)",21
9,"(aminophylline, midazolam)",15


In [43]:
# ============================================================
# STEP 1 - Create Binary DDI Variable
# ============================================================

analysis_df = patient_admission.copy()

analysis_df["DDI"] = (
    (
        analysis_df["num_major"] +
        analysis_df["num_moderate"] +
        analysis_df["num_minor"]
    ) > 0
).astype(int)

print(analysis_df["DDI"].value_counts())

display(
    analysis_df[
        [
            "age",
            "gender",
            "num_drugs",
            "mortality",
            "DDI"
        ]
    ].head()
)

DDI
0    3684
1    3257
Name: count, dtype: int64


,age,gender,num_drugs,mortality,DDI
0,5.19,Female,6,0.0,0
1,5.19,Female,7,0.0,0
2,5.19,Female,6,0.0,0
3,5.19,Female,6,0.0,0
4,5.19,Female,6,0.0,0


In [44]:
# ============================================================
# STEP 2 - Keep Required Columns
# ============================================================

analysis_df = analysis_df[
    [
        "age",
        "gender",
        "num_drugs",
        "mortality",
        "DDI"
    ]
].copy()

print(analysis_df.shape)

display(analysis_df.head())

(6941, 5)


,age,gender,num_drugs,mortality,DDI
0,5.19,Female,6,0.0,0
1,5.19,Female,7,0.0,0
2,5.19,Female,6,0.0,0
3,5.19,Female,6,0.0,0
4,5.19,Female,6,0.0,0


In [45]:
# ============================================================
# STEP 3 - Encode Gender
# ============================================================

analysis_df["gender"] = analysis_df["gender"].map({
    "Male":0,
    "Female":1
})

print(analysis_df["gender"].value_counts())

display(analysis_df.head())

gender
0    4275
1    2666
Name: count, dtype: int64


,age,gender,num_drugs,mortality,DDI
0,5.19,1,6,0.0,0
1,5.19,1,7,0.0,0
2,5.19,1,6,0.0,0
3,5.19,1,6,0.0,0
4,5.19,1,6,0.0,0


In [46]:
# ============================================================
# STEP 4 - Check Missing Values
# ============================================================

print(analysis_df.isnull().sum())

age            0
gender         0
num_drugs      0
mortality    103
DDI            0
dtype: int64


In [47]:
# Remove rows with missing predictor values

analysis_df = analysis_df.dropna()

print("Remaining rows:", len(analysis_df))

Remaining rows: 6838


In [48]:
# ============================================================
# STEP 5 - Multivariate Logistic Regression
# ============================================================

import statsmodels.api as sm

X = analysis_df[
    [
        "age",
        "gender",
        "num_drugs",
        "mortality"
    ]
]

X = sm.add_constant(X)

y = analysis_df["DDI"]

model = sm.Logit(y, X)

result = model.fit()

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.494159
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                    DDI   No. Observations:                 6838
Model:                          Logit   Df Residuals:                     6833
Method:                           MLE   Df Model:                            4
Date:                Wed, 22 Jul 2026   Pseudo R-squ.:                  0.2847
Time:                        06:18:08   Log-Likelihood:                -3379.1
converged:                       True   LL-Null:                       -4723.8
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.3822      0.094    -35.906      0.000      -3.567      -3.198
age            0.0431      0.

In [49]:
# ============================================================
# STEP 6 - Odds Ratio Table
# ============================================================

import numpy as np
import pandas as pd

odds_ratio_table = pd.DataFrame({

    "Odds Ratio": np.exp(result.params),

    "CI Lower": np.exp(result.conf_int()[0]),

    "CI Upper": np.exp(result.conf_int()[1]),

    "P-value": result.pvalues

})

display(odds_ratio_table)

,Odds Ratio,CI Lower,CI Upper,P-value
const,0.033973,0.028245,0.040861,2.469007e-282
age,1.044013,1.029113,1.059128,4.284641e-09
gender,0.713931,0.631372,0.807284,7.688286e-08
num_drugs,1.816059,1.760748,1.873106,0.000000e+00
mortality,1.125942,0.991640,1.278433,6.718787e-02


In [50]:
# Create DDI column
patient_admission["DDI"] = (
    (patient_admission["num_major"] +
     patient_admission["num_moderate"] +
     patient_admission["num_minor"]) > 0
).astype(int)

# Calculate median LOS
median_los_ddi = patient_admission.loc[
    patient_admission["DDI"] == 1, "LOS_days"
].median()

median_los_no_ddi = patient_admission.loc[
    patient_admission["DDI"] == 0, "LOS_days"
].median()

print(f"Median LOS (Patients with DDI): {median_los_ddi:.2f} days")
print(f"Median LOS (Patients without DDI): {median_los_no_ddi:.2f} days")

Median LOS (Patients with DDI): 31.00 days
Median LOS (Patients without DDI): 35.00 days


In [51]:
# Keep one record per unique patient
unique_patients = patient_admission[["subject_id", "gender"]].drop_duplicates(subset="subject_id")

# Total unique patients
total_patients = unique_patients["subject_id"].nunique()

# Number of males and females
gender_counts = unique_patients["gender"].value_counts()

print(f"Total Unique Patients: {total_patients}")
print("\nGender Distribution:")
print(gender_counts)

Total Unique Patients: 709

Gender Distribution:
gender
Male      427
Female    282
Name: count, dtype: int64
